# 03 · Minimum Jerk

### Recap & why now
Notebook 02 ended with a gap: eight coefficients and six conditions leaves two degrees of
freedom, and something has to choose them. "Smoothest" is the obvious answer, and this
notebook makes it precise.

**Jerk** is the third derivative of position — the rate at which acceleration changes.
Minimising it produces the curve Notebook 01 used, and, more importantly, introduces the
machinery that Notebook 04 will point at the fourth derivative instead.

### Learning objectives
1. Name the derivatives up to jerk and say what each means for a vehicle.
2. Write the cost $\int (p^{(3)})^2 dt$ and explain why it is squared.
3. Turn that integral into a **matrix**, and verify it numerically.
4. Derive the minimum-jerk quintic rather than quoting it.
5. Compare it against other curves satisfying the same conditions.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · The derivatives have names

| Derivative | Name | What it means for a drone |
|---|---|---|
| $p$ | position | where it is |
| $\dot p$ | velocity | how fast it moves |
| $\ddot p$ | acceleration | which way it must **tilt** |
| $p^{(3)}$ | **jerk** | how fast the tilt must change → angular rate |
| $p^{(4)}$ | **snap** | how fast the angular rate must change → torque |

Each step down the list lands closer to the actuators, which is why the choice of which
one to minimise is a real engineering decision rather than a matter of taste. This
notebook does jerk; Notebook 04 explains why a quadcopter prefers snap.

In [ ]:
print("  horizontal acceleration   tilt required")
for a_ in (1.0, 3.0, 6.0):
    print("  %20.1f m/s^2 %12.1f°" % (a_, np.degrees(np.arctan(a_/g))))

print("\nnow one level up: to CHANGE the acceleration by 3 m/s^2 (that is jerk), the drone must")
print("change its tilt, and how fast depends on how long you allow:")
for dt in (2.0, 1.0, 0.5, 0.2):
    print("  over %4.1f s -> average angular rate %6.1f deg/s" %
          (dt, np.degrees(np.arctan(3.0/g))/dt))
print("\nEvery derivative you go up, the demand lands closer to the motors. That is the whole")
print("reason for caring which one you minimise.")

## 2 · The cost, and why it is squared

$$J = \int_0^T \left(p^{(3)}(t)\right)^2 dt$$

Squaring is not decoration. Without it, positive and negative jerk cancel: the integral
of $p^{(3)}$ from 0 to $T$ is just $\ddot p(T) - \ddot p(0)$, which depends only on the
endpoints and says nothing about what happened in between. You could wiggle violently
and score zero.

Squaring makes every deviation count, whichever sign it has — and it makes the problem
quadratic, which is what makes it solvable in closed form.

In [ ]:
c_wiggly = np.array([0, 0, 0, 5.0, -12.0, 9.0, -2.0, 0.0])   # A deliberately restless polynomial.
grid = np.linspace(0, 1, 4001)
plain = np.trapezoid([poly_val(c_wiggly, t_, 3) for t_ in grid], grid)
squared = np.trapezoid([poly_val(c_wiggly, t_, 3)**2 for t_ in grid], grid)
print("integral of jerk        : %8.4f   <- only the endpoints matter" % plain)
print("integral of jerk SQUARED: %8.4f   <- every wiggle counts" % squared)

fig, ax = plt.subplots(figsize=(7.4, 2.8))
ax.plot(grid, [poly_val(c_wiggly, t_, 3) for t_ in grid], color="C3", lw=2, label="jerk")
ax.fill_between(grid, [poly_val(c_wiggly, t_, 3) for t_ in grid], color="C3", alpha=0.15)
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("t"); ax.set_ylabel("jerk"); ax.legend(fontsize=9)
ax.set_title("The positive and negative areas nearly cancel — which is the problem")
plt.show()

## 3 · The cost as a matrix

Here is the trick that makes everything computable. $p^{(3)}(t)$ is **linear** in the
coefficients, so its square is **quadratic**, and any quadratic can be written as a
matrix sandwich:

$$J = c^\top Q\, c$$

$Q$ is computed once, in closed form, by expanding the integral term by term. Do not
memorise the formula — write it, then check it against a numerical integral, which is
what the next cell does.

In [ ]:
n, T = 6, 2.5
c_test = np.array([0.3, -1.2, 0.4, 2.0, -0.7, 1.1])
Q3 = cost_matrix(n, T, der=3)                      # der=3 penalises jerk.

analytic = c_test @ Q3 @ c_test
grid = np.linspace(0, T, 200001)
numeric = np.trapezoid([poly_val(c_test, t_, 3)**2 for t_ in grid], grid)
print("jerk cost from the matrix      : %.6f" % analytic)
print("jerk cost by numerical integral: %.6f" % numeric)
print("relative difference            : %.2e  <- the matrix is right" % (abs(analytic-numeric)/numeric))

print("\nQ, rounded:")
print(np.round(Q3, 2))
print("\nThe first three rows and columns are empty. c0, c1 and c2 describe the segment's")
print("starting position, velocity and acceleration — none of which bends the curve, so none of")
print("which costs any jerk. Only c3 and above are penalised.")

## 4 · Deriving the quintic

Now put the pieces together. Six boundary conditions pin down six coefficients exactly,
so for a single rest-to-rest segment there is **no freedom left** — the minimum-jerk
answer is forced.

$$s(\tau) = 10\tau^3 - 15\tau^4 + 6\tau^5$$

The cost matrix earns its keep in Notebook 05, where the number of coefficients exceeds
the number of conditions and something has to choose. Here we simply confirm that the
familiar curve is the one the conditions produce.

In [ ]:
def solve_exact(conditions, n):
    A = np.array([row for row, _ in conditions]); b = np.array([val for _, val in conditions])
    return np.linalg.solve(A, b)

T = 1.0
six = [(deriv_row(6, 0.0, d), 0.0) for d in range(3)] + \
      [(deriv_row(6, T, d), 1.0 if d == 0 else 0.0) for d in range(3)]
c_mj = solve_exact(six, 6)
print("minimum-jerk coefficients:", np.round(c_mj, 6))
print("that is 10t^3 - 15t^4 + 6t^5 ✔  jerk cost = %.2f" % (c_mj @ cost_matrix(6, T, 3) @ c_mj))

alternatives = {"minimum jerk (quintic)": c_mj,
                "cubic (no accel conditions)": solve_exact(
                    [(deriv_row(4, 0.0, 0), 0.0), (deriv_row(4, 0.0, 1), 0.0),
                     (deriv_row(4, T, 0), 1.0), (deriv_row(4, T, 1), 0.0)], 4)}
print("\n  curve                          jerk cost   peak accel")
grid = np.linspace(0, T, 1000)
for name, cc in alternatives.items():
    Qc = cost_matrix(len(cc), T, 3)
    peak_a = max(abs(poly_val(cc, t_, 2)) for t_ in grid)
    print("  %-30s %10.2f %11.3f" % (name, cc @ Qc @ cc, peak_a))
print("\nThe cubic is CHEAPER on jerk — because it was never asked to have zero acceleration at")
print("the ends. It starts and stops with an acceleration step instead, which for a quadcopter")
print("means asking for an instantaneous tilt. Fewer constraints, lower cost, worse trajectory.")

## 🧪 Try it yourself

**E1.** The cubic scored a lower jerk cost than the quintic. Does that make it a better
trajectory? What is the cost failing to capture?

**E2.** Compute the jerk cost of the quintic as the duration $T$ changes and work out
the power law. Predict it from the units before you measure.

In [ ]:
# --- Solution E1 ---
grid = np.linspace(0, 1.0, 1000)
c_cubic = alternatives["cubic (no accel conditions)"]
print("E1: no. The two curves answer DIFFERENT questions, so their costs are not comparable.")
print("    The cubic's acceleration jumps from 0 to %.2f m/s^2 at t = 0 —" % poly_val(c_cubic, 0.0, 2))
print("    an instantaneous change, which for a quadcopter is an instantaneous TILT. The quintic")
print("    starts at %.2f." % poly_val(c_mj, 0.0, 2))
print("    A cost only measures what you told it to measure. Adding constraints always raises the")
print("    achievable minimum, so a lower number can simply mean a laxer specification — and that")
print("    is a trap worth remembering when comparing any two optimisation results.")

# --- Solution E2 ---
print("\nE2: prediction from units — jerk is a third derivative, so it scales as 1/T^3; squaring")
print("    gives 1/T^6; integrating over a duration T multiplies by T. Net: 1/T^5.")
print("\n     T [s]    jerk cost    ratio to previous")
prev = None
for T_ in (0.5, 1.0, 2.0, 4.0):
    six_T = [(deriv_row(6, 0.0, d), 0.0) for d in range(3)] + \
            [(deriv_row(6, T_, d), 1.0 if d == 0 else 0.0) for d in range(3)]
    cc = solve_exact(six_T, 6)
    J = cc @ cost_matrix(6, T_, 3) @ cc
    ratio = "" if prev is None else "%.3f" % (J/prev)
    print("    %6.1f %12.3f %14s" % (T_, J, ratio))
    prev = J
print("    Each doubling divides the cost by %.0f = 2^5 ✔ — the prediction holds exactly." % 32)
print("    Practical reading: flying a manoeuvre half as fast is thirty-two times gentler on")
print("    whatever the cost is measuring. Notebook 08 turns that into a design tool.")

## 🚁 Mini-project: comparing curves that meet the same conditions

Animate a drone following the quintic and the cubic side by side, with the tilt each
demands shown underneath. Both start and end in the same place at the same time; only
the shape of the demand differs.

In [ ]:
T, dist = 2.0, 3.0
times = np.linspace(0, T, 140)
curves = {"minimum jerk": (c_mj, 6), "cubic": (c_cubic, 4)}
traces = {}
for name, (cc, _) in curves.items():
    traces[name] = np.array([[dist*poly_val(cc, t_/T, d)/T**d for d in range(3)] for t_ in times])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 4.6), gridspec_kw={"height_ratios": [2, 1]})

def frame(k):
    a1.clear(); a2.clear()
    for (name, tr), col, y in zip(traces.items(), ["C0", "C3"], [1, 0]):
        a1.plot([0, dist], [y, y], color="0.85", lw=2)
        a1.plot(tr[k, 0], y, "o", color=col, ms=12)
        a1.text(-0.7, y, name, fontsize=9, va="center", color=col)
        a2.plot(times[:k+1], np.degrees(np.arctan(tr[:k+1, 2]/g)), color=col, lw=1.7)
    a1.set_xlim(-1.0, 3.4); a1.set_ylim(-0.6, 1.6); a1.set_yticks([]); a1.set_xlabel("position [m]")
    a1.set_title("t = %4.2f s" % times[k], fontsize=10)
    a2.set_xlim(0, T); a2.set_ylim(-20, 20)
    a2.set_xlabel("time [s]"); a2.set_ylabel("tilt demanded [deg]")
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Minimum-jerk trajectories come originally from human motor
> control research — Flash and Hogan showed in 1985 that human arm movements closely
> follow a minimum-jerk profile. Robotics adopted it for the same reason a quadcopter
> wants it: smooth in the derivatives that reach the actuators. The choice of *which*
> derivative to minimise is what changes between applications.

**Where next.** For a quadcopter, jerk is not quite the right derivative. Notebook 04
follows the chain from position all the way down to motor thrust and finds that the
fourth derivative is the one that matters.